In [3]:
from __future__ import annotations
import warnings

import numpy as np
import pandas as pd
from scipy.stats import norm, spearmanr

warnings.filterwarnings("ignore")


# ─────────────────────────────────────────────────────────────────────────────
# 0.  RETURN GENERATOR 
# ─────────────────────────────────────────────────────────────────────────────

def generate_returns(
    T: int,
    mu: float,
    sigma: float,
    skew: float,
    kurt_excess: float,
    seed: int | None = None,
) -> np.ndarray:
    """
    T iid returns with approximate 4-moment targets via Cornish-Fisher:
        w = z + (s/6)(z²-1) + (k/24)(z³-3z) - (s²/36)(2z³-5z)
    Re-standardised to exact mean=0, std=1 before applying (mu, sigma).
    Replace with your own generator.
    """
    rng = np.random.default_rng(seed)
    z   = rng.standard_normal(T)
    s, k = skew, kurt_excess
    w = (
        z
        + (s / 6)  * (z**2 - 1)
        + (k / 24) * (z**3 - 3 * z)
        - (s**2 / 36) * (2 * z**3 - 5 * z)
    )
    w = (w - w.mean()) / w.std(ddof=1)
    return mu + sigma * w


# ─────────────────────────────────────────────────────────────────────────────
# 1.  PORTFOLIO PARAMETERS
# ─────────────────────────────────────────────────────────────────────────────


sr_star = 0.1

PORTFOLIO_PARAMS = [
    # name       mu     sigma   skew    excess_kurt
    ("P1",    sr_star+0.101,   1,    -1.50,  2.00),
    ("P2",    sr_star+0.100,   1,     0.00,  0.00),
    ("P3",    sr_star+0.098,   1,    -2.00,  10.0),
    ("P4",    sr_star+0.097,   1,    -0.00,  0.00),
    ("P5",    sr_star+0.095,   1,    -3.00,  10.00),
    ("P6",    sr_star+0.094,   1,    -0.00,  0.00),
    ("P7",    sr_star+0.092,   1,    -4.0,  18.00),
    ("P8",    sr_star+0.091,   1,    -0.00,  0.00),
]

T    = 50
SEED = 42


# ─────────────────────────────────────────────────────────────────────────────
# 2.  METRIC FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def _sample_moments(r: np.ndarray) -> tuple[float, float, float]:
    """(SR, skewness, excess kurtosis) from a return array."""
    s  = pd.Series(r)
    return float(s.mean() / s.std(ddof=1)), float(s.skew()), float(s.kurt())


def psr_normal(r: np.ndarray, sr_star: float = 0.0) -> float:
    """
    PSR under NAIVE NORMAL assumption  (γ₁ = γ₂ = 0).

    Asymptotic variance simplifies to:
        V̂_normal = (1/(T-1)) · (1 + SR̂²/2)

    PSR_normal = Φ[ (SR̂ − SR*) / √V̂_normal ]

    This is the standard formula used when higher moments are ignored.
    It underestimates the true variance for negatively-skewed / fat-tailed
    series → overestimates the probability that SR > SR*.
    """
    T  = len(r)
    sr, _, _ = _sample_moments(r)
    V  = (1.0 / (T - 1)) * (1.0 + sr**2 / 2.0)
    return float(norm.cdf((sr - sr_star) / np.sqrt(V)))


def psr_general(r: np.ndarray, sr_star: float = 0.0) -> float:
    """
    PSR under GENERAL IID DGP  (Lo 2002 asymptotic variance).

    Asymptotic variance:
        V̂_general = (1/(T-1)) · (1 − γ₁·SR̂ + (γ₂+2)/4·SR̂²)

    For γ₁ < 0 (negative skew) the −γ₁·SR̂ > 0 term increases V̂,
    widening the confidence interval and reducing PSR relative to PSR_normal.
    Kurtosis γ₂ > 0 further inflates V̂ through the (γ₂+2)/4·SR̂² term.

    PSR_general = Φ[ (SR̂ − SR*) / √V̂_general ]
    """
    T  = len(r)
    sr, sk, ku = _sample_moments(r)
    V  = (1.0 / (T - 1)) * (1.0 - sk * sr + ((ku + 2.0) / 4.0) * sr**2)
    if V <= 1e-10:
        return np.nan
    return float(norm.cdf((sr - sr_star) / np.sqrt(V)))


def adjusted_sharpe_ratio(r: np.ndarray) -> float:
    """
    ASR = SR · [1 + (γ₁/6)·SR − (γ₂/24)·SR²]  (Pézier & White 2006).

    Direct polynomial correction to SR. Negative γ₁ and positive γ₂
    both reduce ASR relative to SR. With σ = 1 and yearly observations,
    SR is already on the native scale — no annualisation needed.
    """
    sr, sk, ku = _sample_moments(r)
    return float(sr * (1.0 + (sk / 6.0) * sr - (ku / 24.0) * sr**2))


# ─────────────────────────────────────────────────────────────────────────────
# 3.  GENERATE SERIES & COMPUTE METRICS
# ─────────────────────────────────────────────────────────────────────────────

records: list[dict] = []

for idx, (name, mu, sigma, skew, kurt_ex) in enumerate(PORTFOLIO_PARAMS):
    r = generate_returns(T, mu, sigma, skew, kurt_ex, seed=SEED + idx)
    sr, sk_r, ku_r = _sample_moments(r)

    # Theoretical V under each assumption (for transparency)
    V_n = (1.0 / (T - 1)) * (1.0 + sr**2 / 2.0)
    V_g = (1.0 / (T - 1)) * (1.0 - sk_r * sr + ((ku_r + 2.0) / 4.0) * sr**2)

    psr_n = psr_normal(r, sr_star=sr_star)
    psr_g = psr_general(r, sr_star=sr_star)
    asr   = adjusted_sharpe_ratio(r)

    records.append({
        "Portfolio"  : name,
        "SR"         : sr,
        "Skew"       : sk_r,
        "ExKurt"     : ku_r,
        "V_normal"   : V_n,
        "V_general"  : V_g,
        "V_ratio"    : V_g / V_n,          # > 1 means general is more conservative
        "PSR_normal" : psr_n,
        "PSR_general": psr_g,
        "PSR_gap"    : psr_n - psr_g,      # overconfidence of the normal assumption
        "Adj_SR"     : asr,
    })

df = pd.DataFrame(records)


# ─────────────────────────────────────────────────────────────────────────────
# 4.  RANKINGS  (rank 1 = best)
# ─────────────────────────────────────────────────────────────────────────────

df["Rk_PSR_n"] = df["PSR_normal"].rank(ascending=False).astype(int)
df["Rk_PSR_g"] = df["PSR_general"].rank(ascending=False).astype(int)
df["Rk_AdjSR"] = df["Adj_SR"].rank(ascending=False).astype(int)


# ─────────────────────────────────────────────────────────────────────────────
# 5.  DISPLAY
# ─────────────────────────────────────────────────────────────────────────────

W    = 110
SEP  = "=" * W
sep2 = "-" * W

print(SEP)
print(f"  PSR (iid Normal) vs PSR (iid General) vs Adjusted SR — T={T}, SR*={sr_star}")
print(SEP)

# ── Main table ───────────────────────────────────────────────────────────────
hdr = (
    f"{'Portfolio':>10}  {'SR':>6}  {'Skew':>6}  {'Kurt':>7}  "
    f"{'PSR_norm':>9}  {'PSR_gen':>8}  {'Gap':>6}  {'Adj_SR':>7}"
)
print(hdr)
print(sep2)

for _, row in df.sort_values("Portfolio").iterrows():
    print(
        f"{row['Portfolio']:>10}  "
        f"{row['SR']:>6.4f}  "
        f"{row['Skew']:>+6.2f}  "
        f"{3+row['ExKurt']:>7.2f}  "
        f"{row['PSR_normal']:>9.4f}  "
        f"{row['PSR_general']:>8.4f}  "
        f"{row['PSR_gap']:>+6.4f}  "
        f"{row['Adj_SR']:>7.4f}"
    )

print(sep2)
print("  Gap = PSR_norm − PSR_gen  (positive = normal assumption is over-confident)")

# ── Rankings ─────────────────────────────────────────────────────────────────
print()
print("Rankings  (1 = best)")
print(sep2)
hdr2 = f"{'Portfolio':>10}  {'SR':>6}  {'Rk_PSR_norm':>12}  {'Rk_PSR_gen':>11}  {'Rk_AdjSR':>9}"
print(hdr2)
print(sep2)
for _, row in df.sort_values("Portfolio").iterrows():
    n_vs_g = int(row["Rk_PSR_n"] - row["Rk_PSR_g"])
    print(
        f"{row['Portfolio']:>10}  "
        f"{row['SR']:>6.4f}  "
        f"{int(row['Rk_PSR_n']):>12}  "
        f"{int(row['Rk_PSR_g']):>11}  "
        f"{int(row['Rk_AdjSR']):>9}"
    )
print(sep2)

# ── Spearman correlations ─────────────────────────────────────────────────────
print()
print("Spearman rank correlations")
print(sep2)
pairs = [
    ("PSR_gen", "Rk_PSR_g", "Adj_SR",   "Rk_AdjSR",),
    #("PSR_norm","Rk_PSR_n", "PSR_gen",  "Rk_PSR_g",),
    ("PSR_norm","Rk_PSR_n", "Adj_SR",   "Rk_AdjSR",),
]
for a, ca, b, cb in pairs:
    rho, _ = spearmanr(df[ca], df[cb])
    print(f"  {a:9} <-> {b:9} :  rho = {rho:+.3f}")
print(sep2)

  PSR (iid Normal) vs PSR (iid General) vs Adjusted SR — T=50, SR*=0.1
 Portfolio      SR    Skew     Kurt   PSR_norm   PSR_gen     Gap   Adj_SR
--------------------------------------------------------------------------------------------------------------
        P1  0.2010   -1.17     4.37     0.7580    0.7348  +0.0232   0.1926
        P2  0.2000   -0.17     2.57     0.7559    0.7528  +0.0030   0.1990
        P3  0.1980   -2.39    13.40     0.7515    0.7066  +0.0450   0.1790
        P4  0.1970   -0.03     2.57     0.7494    0.7493  +0.0001   0.1970
        P5  0.1950   -1.58     5.14     0.7450    0.7166  +0.0284   0.1843
        P6  0.1940   -0.12     2.29     0.7428    0.7411  +0.0017   0.1935
        P7  0.1920   -2.09     7.71     0.7383    0.7028  +0.0355   0.1778
        P8  0.1910   -0.23     3.11     0.7361    0.7317  +0.0043   0.1896
--------------------------------------------------------------------------------------------------------------
  Gap = PSR_norm − PSR_gen  (posi